# Validação do Pipeline — FSCIL-Lab

Diagnóstico Fitossanitário Incremental com Descritores Clássicos de Imagem

Este notebook valida cada etapa do pipeline: pré-processamento, extração de características,
memória de protótipos e execução do protocolo FSCIL.

In [ ]:
import sys, os
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

caminho_projeto = Path(os.getcwd()).parent
sys.path.insert(0, str(caminho_projeto))

from src.dataset_loader import carregar_dataset_bruto, dividir_em_sessoes, preparar_dados_sessao
from src.preprocessing import pipeline_completo_preprocessamento
from src.feature_extraction import extrair_descritores_glcm, extrair_momentos_hu, extrair_descritores_morfologicos, extrair_descritores_hog, extrair_todas_caracteristicas
from src.prototype_memory import MemoriaDePrototipos
from src.metrics import calcular_acuracia, montar_matriz_confusao, gerar_relatorio_sessoes
from src.session_manager import GerenciadorDeSessoes

## 1. Geração de dados sintéticos para validação

In [ ]:
def gerar_folhas_sinteticas(quantidade, altura=64, largura=64):
    """Gera imagens sintéticas simulando folhas com texturas diferentes."""
    imagens = np.zeros((quantidade, altura, largura, 3), dtype=np.uint8)
    for i in range(quantidade):
        fundo = np.random.randint(30, 60)
        imagens[i] = fundo
        centro_y, centro_x = np.random.randint(10, altura-10), np.random.randint(10, largura-10)
        Y, X = np.ogrid[:altura, :largura]
        mascara = (X - centro_x)**2 + (Y - centro_y)**2 < (np.random.randint(10, 25))**2
        if i < quantidade // 2:
            imagens[i, mascara] = [np.random.randint(80, 120), np.random.randint(120, 160), np.random.randint(40, 80)]
        else:
            imagens[i, mascara] = [np.random.randint(150, 200), np.random.randint(80, 120), np.random.randint(30, 60)]
    return imagens

imagens_treino = gerar_folhas_sinteticas(100)
rotulos_treino = np.array([i // 10 for i in range(100)], dtype=np.int64)
imagens_teste = gerar_folhas_sinteticas(50)
rotulos_teste = np.array([i // 5 for i in range(50)], dtype=np.int64)

print(f'Treino: {imagens_treino.shape}, {len(np.unique(rotulos_treino))} classes')
print(f'Teste: {imagens_teste.shape}, {len(np.unique(rotulos_teste))} classes')

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for i in range(5):
    axes[0, i].imshow(imagens_treino[i])
    axes[0, i].set_title(f'Treino Classe {rotulos_treino[i]}')
    axes[0, i].axis('off')
    axes[1, i].imshow(imagens_treino[50 + i])
    axes[1, i].set_title(f'Treino Classe {rotulos_treino[50 + i]}')
    axes[1, i].axis('off')
plt.tight_layout()
plt.show()

## 2. Pré-processamento

In [ ]:
preprocessado = pipeline_completo_preprocessamento(imagens_treino[:5], tamanho_alvo=(64, 64))
print('Imagens cinza:', preprocessado['imagens_cinza'].shape)
print('Normalizadas:', preprocessado['imagens_normalizadas'].shape)
print('Máscaras:', preprocessado['mascaras_binarias'].shape)

fig, axes = plt.subplots(3, 5, figsize=(15, 9))
for i in range(5):
    axes[0, i].imshow(preprocessado['imagens_cinza'][i], cmap='gray')
    axes[0, i].set_title('Cinza')
    axes[0, i].axis('off')
    axes[1, i].imshow(preprocessado['imagens_normalizadas'][i], cmap='gray')
    axes[1, i].set_title('Normalizada')
    axes[1, i].axis('off')
    axes[2, i].imshow(preprocessado['mascaras_binarias'][i], cmap='gray')
    axes[2, i].set_title('Sauvola')
    axes[2, i].axis('off')
plt.tight_layout()
plt.show()

## 3. Extração de características

In [ ]:
preprocessado_treino = pipeline_completo_preprocessamento(imagens_treino, tamanho_alvo=(64, 64))
caracteristicas, escalonador = extrair_todas_caracteristicas(
    preprocessado_treino['imagens_cinza'],
    preprocessado_treino['mascaras_binarias'],
    incluir_hog=True,
)
print(f'Matriz de características: {caracteristicas.shape}')
print(f'Dimensão do vetor por imagem: {caracteristicas.shape[1]}')
print(f'Média (deve ser ~0): {caracteristicas.mean():.6f}')
print(f'Desvio padrão médio (deve ser ~1): {caracteristicas.std(axis=0).mean():.6f}')

## 4. Memória de protótipos (NCM)

In [ ]:
memoria = MemoriaDePrototipos()
memoria.adicionar_classes(caracteristicas, rotulos_treino)
print(f'Protótipos: {memoria.prototipos.shape}')

preprocessado_teste = pipeline_completo_preprocessamento(imagens_teste, tamanho_alvo=(64, 64))
caracteristicas_teste, _ = extrair_todas_caracteristicas(
    preprocessado_teste['imagens_cinza'],
    preprocessado_teste['mascaras_binarias'],
    incluir_hog=True,
    escalonador=escalonador,
)
predicoes = memoria.classificar(caracteristicas_teste)
acuracia = calcular_acuracia(rotulos_teste, predicoes)
print(f'Acurácia no teste: {acuracia:.4f}')
print(f'Matriz de confusão:\n{montar_matriz_confusao(rotulos_teste, predicoes)}')

## 5. Simulação FSCIL completa

In [ ]:
dados_brutos = {
    'imagens': np.concatenate([imagens_treino, imagens_teste], axis=0),
    'rotulos': np.concatenate([rotulos_treino, rotulos_teste], axis=0),
    'nomes_classes': [f'Classe_{i}' for i in range(10)],
}
sessoes = dividir_em_sessoes(dados_brutos['rotulos'], 6, 2, 42)
print('Sessão base:', sessoes['sessao_base'])
print('Sessões:', sessoes['sessoes_incrementais'])

gerenciador = GerenciadorDeSessoes(dados_brutos, sessoes, tamanho_alvo=(64, 64), incluir_hog=True)
resultado_base = gerenciador.executar_sessao_base()
print(f'Acurácia base: {resultado_base["metricas"]:.4f}')

for i in range(len(sessoes['sessoes_incrementais'])):
    resultado = gerenciador.executar_sessao_incremental(i, quantidade_por_classe=3)
    print(f'Sessão {i+1}: acurácia = {resultado["metrica"]:.4f}')

relatorio = gerenciador.obter_relatorio()
print(f'\nAcurácia média: {relatorio["acuracia_media"]:.4f}')
print(f'PD: {relatorio["performance_dropping_rate"]:.4f}')
print(f'Forgetting: {relatorio["esquecimento_medio"]:.4f}')

# Plotar curva de acurácia
plt.figure(figsize=(8, 4))
plt.plot(range(len(relatorio['historico_acuracias'])), relatorio['historico_acuracias'], marker='o', linewidth=2)
plt.xlabel('Sessão')
plt.ylabel('Acurácia')
plt.title('Acurácia por Sessão (FSCIL)')
plt.grid(True, alpha=0.3)
plt.ylim(0, 1)
plt.show()

## Conclusão

O pipeline completo funciona conforme o esperado:
- Pré-processamento com redimensionamento, normalização e Sauvola
- Extração de características (GLCM, Hu, morfologia, HOG opcional)
- Classificação NCM com memória de protótipos incremental
- Protocolo FSCIL com sessão base e sessões incrementais
- Métricas de avaliação (acurácia, PD, forgetting)